In [ ]:
from static_bilevel import StaticBilevelModel, FruitInfo
import numpy as np
import time

# Example usage
height = 1.2
length = 1.2
arms_n = 2
vx_max = 0.5
vy_max = 0.5
vz_max = 0.5
ax_max = 1.0
ay_max = 1.0
az_max = 1.0
dt = 0.1
safety_distance = 0.0
seed_num = 100
infeasibility_count = 0
for seed in range(seed_num):
    # Generate random fruits for testing
    rng = np.random.default_rng(seed)
    fruit_n = 8
    depths = [0.2, 0.5]
    X = rng.uniform(0.0, length, fruit_n) # length
    # X = np.array([0.4, 0.5, 0.6, 0.7])
    Y = rng.uniform(depths[0], depths[1], fruit_n)  # depth
    Z = rng.uniform(safety_distance, height - safety_distance, fruit_n)  # height
    # Z = np.array([0.5, 0.7, 0.5, 0.7])
    detach_times = rng.uniform(2.0, 3.0, fruit_n)  # detach times, it could be a function of ripening levels or it could be a constant.
    fruits = [FruitInfo(X[i], Y[i], Z[i], detach_times[i]) for i in range(fruit_n)]
    model = StaticBilevelModel(fruits, height, length, arms_n, vx_max, vy_max, vz_max, ax_max, ay_max, az_max, dt, safety_distance)
    # model.plot_initial_fruits_and_arms()
    tic = time.time()
    try:
        # model.bilevel_trajectory_planning() # Not working for now
        model.zone_splitting_planning(gap=0.05) # Larger gap for faster solving
    except Exception as e:
        print(f"Optimization failed for seed {seed} with error: {e}")
        infeasibility_count += 1
        continue
    toc = time.time()
    print(f"Time taken for optimization: {toc - tic:.3f} seconds")
    print("Optimization completed.")
    # model.draw_trajectories(video=True, show_trajectory=False)
    print(f"Model results are:\n{model.arms_plan}")
    print(f"Trajectories are stored in model.arms_plan.paths_x, paths_y, paths_z.")
print(f"Number of infeasible optimizations: {infeasibility_count} out of {seed_num}")